# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [1]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


## Ejemplo

In [2]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [3]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [4]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [5]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [6]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.118887,0.158965,0.991222,0.695055,0.982407,0.018471,0.000642,0.007697,0.000904
TensorizedQDA,0.121517,0.081330,0.442137,0.330881,0.982593,0.018471,0.000680,0.012123,0.000174


In [8]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.118887,0.991222,0.982407
TensorizedQDA,0.121517,0.442137,0.982593


In [9]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.118887,0.158965,0.991222,0.695055,0.982407,0.018471,0.000642,0.007697,0.000904,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.121517,0.081330,0.442137,0.330881,0.982593,0.018471,0.000680,0.012123,0.000174,0.978357,2.241889,1.0,0.634912


In [10]:
# volvemos a subsetear columnas
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.118887,0.991222,0.982407,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.121517,0.442137,0.982593,0.978357,2.241889,1.0,0.634912


In [11]:
# Levantamos los datasets Iris, Letters, Penguins y Wine
X_full_iris, y_full_iris = get_iris_dataset()
# X_full_letters, y_full_letters = get_letters_dataset()
# X_full_penguins, y_full_penguins = get_penguins_dataset()
X_full_wine, y_full_wine = get_wine_dataset()

X_full_iris.shape, y_full_iris.shape
# X_full_letters.shape, y_full_letters.shape
# X_full_penguins.shape, y_full_penguins.shape
X_full_wine.shape, y_full_wine.shape

((178, 13), (178, 1))

In [12]:
# Encodeamos a número las clases
y_full_iris_encoded = label_encode(y_full_iris)
y_full_wine_encoded = label_encode(y_full_wine)

y_full_iris[:5], y_full_iris_encoded[:5]
y_full_wine[:5], y_full_wine_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [13]:
benchmark_iris = Benchmark(
    X_full_iris, y_full_iris_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

benchmark_wine = Benchmark(
    X_full_wine, y_full_wine_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [14]:
# Benchmark de los modemos en el dataset Iris
to_bench = [QDA, TensorizedQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    benchmark_iris.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [15]:
for model in to_bench:
    benchmark_wine.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [16]:
summary_iris = benchmark_iris.summary(baseline='QDA')
summary_iris[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.103662,0.722292,0.970667,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.080636,0.262414,0.971111,1.285555,2.752490,1.011858,0.878836
QDA_Chol1,0.126707,0.483468,0.972889,0.818124,1.493981,0.965461,0.947228
QDA_Chol2,0.106502,1.082658,0.972222,0.973338,0.667147,1.015662,0.934053
QDA_Chol3,0.109842,0.496578,0.977333,0.943737,1.454539,0.998768,1.020702


In [17]:
summary_wine = benchmark_wine.summary(baseline='QDA')
summary_wine[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.116576,0.982045,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.118447,0.439307,0.982593,0.984208,2.235442,1.000000,0.632599
QDA_Chol1,0.147387,0.582619,0.985741,0.790955,1.685569,1.023799,0.976240
QDA_Chol2,0.114987,1.328272,0.983333,1.013823,0.739341,1.029487,0.951586
QDA_Chol3,0.118772,0.602655,0.986111,0.981515,1.629532,1.032134,0.994807


# Consigna QDA

## Tensorización

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

> **R:** Paraleliza únicamente sobre las $k$ clases, ya que la cadena de llamadas es: `predict(X) -> predict_one(X) -> predict_log_conditionals(X)` donde X($p$, 1) es una sola observación. La iteración sobre las $n$ observaciones a predecir se sigue haciendo en `fiuba-ceia-amia-tp/base/bayesian.py` en la línea 34.

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.
> **R:** La forma de `tensor_inv_covs` es ($k$, $p$, $p$) y la forma de `tensor_means` es ($k$, $p$, 1). Para predecir, `TensorizedQDA` hace un producto punto entre `tensor_inv_covs` y `tensor_means`. La razón principal por la que predicen lo mismo está en observar el código de `_predict_log_conditional` y `_predict_log_conditionals`: ambas devuelven el cálculo de: $$f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}$$ La única diferencia es que `QDA` lo hace para una clase a la vez, y `TensorizedQDA` lo hace para todas las clases en paralelo. En esta misma línea, en ambas clases el método `_predict_one` devuelve el índice de la clase que maximiza la probabilidad condicional a posteriori: $$log P(G=k) + log P(x|G=k)$$ Por lo tanto, ambas clases predicen lo mismo, la diferencia es únicamente en performance, siendo `TensorizedQDA` más rápido al paralelizar sobre las clases.

In [18]:
from base.qda import TensorizedQDA
from utils.datasets import get_wine_dataset, label_encode
from sklearn.model_selection import train_test_split

X, y = get_wine_dataset()
y = label_encode(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# transponer porque la clase espera (n_feat, n_obs)
X_train = X_train.T
y_train = y_train.T

model = TensorizedQDA()
model.fit(X_train, y_train)

print('tensor_inv_cov:', model.tensor_inv_cov.shape)
print('tensor_means:  ', model.tensor_means.shape)


tensor_inv_cov: (3, 13, 13)
tensor_means:   (3, 13, 1)


### 2) Optimización

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
> R: El código implementado se encuentra en [qda.py](base/qda.py#L47-L58).

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
> **R:** La matriz de $n \times n$ aparece en [qda.py](base/qda.py#L51), donde se calcula `inner_prod`. En esta línea, `unbiased_X` tiene forma ($k$, $p$, $n$) y `tensor_inv_cov` tiene forma ($k$, $p$, $p$). Al hacer el producto punto entre estas dos matrices, se obtiene una matriz de forma ($k$, $n$, $n$).

5. Demostrar que $$diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente $$np.sum(A^T \odot B, axis=0).T$$ queda a preferencia del alumno cuál usar.
> **R:** La demostración parte de como está definido el producto matricial $$C = A \cdot B$$ donde $c_{ij} = \sum_{j} a_{ik} b_{kj}$. Para obtener la diagonal de $C$, se necesita que $i=j$, por lo tanto cada elemento de la diagonal tiene la forma $c_{ii} = \sum_{k} a_{ik} b_{ki}$. Luego, $C' = A \odot B^T$ es un producto elemento a elemento, donde cada elemento tiene la forma $c'_{ik} = a_{ik} b^T_{ik} = a_{ik} b_{ki}$. Vemos entonces la relación, que la sumatoria de los elementos de la fila $i$ de $C'$ es exactamente igual a $c_{ii}$, quedando demostrado que $$diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)$$

In [19]:
import numpy as np

n = 3

matrix_A = np.random.rand(n, n)
matrix_B = np.random.rand(n, n)

diag_product = np.diag(matrix_A @ matrix_B)

print('Diagonal del producto de A y B:', diag_product)

sum_cols = np.sum(matrix_A * matrix_B.T, axis=1)
print('Suma de columnas de A * B^T:', sum_cols)

sum_cols_equiv = np.sum(matrix_A.T * matrix_B, axis=0).T
print('Suma de columnas de A^T * B (equivalente):', sum_cols_equiv)

Diagonal del producto de A y B: [0.36108931 0.90412248 1.05724578]
Suma de columnas de A * B^T: [0.36108931 0.90412248 1.05724578]
Suma de columnas de A^T * B (equivalente): [0.36108931 0.90412248 1.05724578]


6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.
> R: El código implementado se encuentra en [qda.py](base/qda.py#L59-L71).

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?
> **R:** Se coincide con lo esperado: `EfficientQDA > FasterQDA > TensorizedQDA > QDA` en términos de iteraciones por segundo en cuanto a la inferencia del modelo. Del análisis se desprende que todos los modelos se entrenan más o menos a la misma velocidad, la eficiencia es únicamente en términos de predicción. Esto tiene sentido, ya que las clases implementadas solo aplican sobreescriben los métodos de inferencia.

In [26]:
from base.qda import FasterQDA, EfficientQDA

tensorized_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in tensorized_bench:
    benchmark_iris.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [27]:
tensorized_summary_iris = benchmark_iris.summary(baseline='QDA')
tensorized_summary_iris[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.112457,0.751329,0.972889,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.101582,0.269960,0.974889,1.107056,2.783112,1.008195,0.896805
QDA_Chol1,0.126707,0.483468,0.972889,0.887536,1.554041,0.952360,0.935190
QDA_Chol2,0.106502,1.082658,0.972222,1.055919,0.693967,1.001879,0.922182
QDA_Chol3,0.109842,0.496578,0.977333,1.023807,1.513013,0.985215,1.007731
FasterQDA,0.075191,0.015925,0.974000,1.495608,47.177735,1.008831,0.063879
EfficientQDA,0.073412,0.015621,0.973778,1.531872,48.098901,1.002507,0.185450


In [28]:
for model in tensorized_bench:
    benchmark_wine.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [29]:
tensorized_summary_wine = benchmark_wine.summary(baseline='QDA')
tensorized_summary_wine[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.119917,0.992876,0.981481,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.123822,0.446037,0.981667,0.968463,2.225994,0.982606,0.634468
QDA_Chol1,0.147387,0.582619,0.985741,0.813620,1.704158,0.997848,0.979126
QDA_Chol2,0.114987,1.328272,0.983333,1.042874,0.747494,1.003392,0.954399
QDA_Chol3,0.118772,0.602655,0.986111,1.009640,1.647502,1.005972,0.997748
FasterQDA,0.093761,0.023786,0.983889,1.278965,41.742886,0.996366,0.069693
EfficientQDA,0.089376,0.021666,0.985000,1.341706,45.827489,1.007266,0.101419
